# Grouped-Query Attention (GQA) — Interactive Demo

This notebook walks through the full GQA implementation end-to-end:

| Section | Topic |
|---------|-------|
| 1 | KV-Cache memory math |
| 2 | `repeat_kv` broadcasting utility |
| 3 | `RotaryEmbedding` (RoPE) |
| 4 | Full `GroupedQueryAttention` module (GQA + RoPE + causal mask) |
| 5 | Model surgery: MHA → GQA via mean-pooling |
| 6 | Latency benchmark: MHA vs GQA vs MQA |

**Paper:** [GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints](https://arxiv.org/abs/2305.13245) (Ainslie et al., 2023)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn as nn
from gqa.attention import repeat_kv, RotaryEmbedding, GroupedQueryAttention
from gqa.surgery   import convert_mha_to_gqa, memory_comparison_table
from benchmarks.benchmark import run_benchmark

torch.manual_seed(42)
print('PyTorch version:', torch.__version__)

---
## Section 1 — KV-Cache Memory Math

The KV-cache stores **K** and **V** tensors for every generated token. The size of one layer's cache is:

$$\text{bytes} = 2 \times B \times T \times H_{KV} \times d_{head} \times \text{bytes\_per\_element}$$

where the leading `2` accounts for both K and V.

In [ ]:
rows = memory_comparison_table(num_query_heads=32, seq_len=4096, batch_size=1, d_model=4096)

print(f"{'Architecture':<12} {'H_Q':>5} {'H_KV':>6} {'KV Cache (MB)':>14} {'Reduction':>11}")
print('-' * 52)
for r in rows:
    print(f"{r['Architecture']:<12} {r['H_Q']:>5} {r['H_KV']:>6} "
          f"{r['KV Cache (MB)']:>14.1f} {r['Reduction']:>11}")

---
## Section 2 — `repeat_kv`: Broadcasting Without Copying Memory

GQA has **fewer KV heads than Q heads**. Before computing attention, we need to
expand the KV tensors so their head dimension matches Q — ideally *without*
allocating new memory.

We achieve this with `.expand()` (a view operation) followed by `.reshape()`.

In [ ]:
# Simulating 2 KV heads that need to serve 4 Q heads (repeat_factor = 2)
B, H_kv, T, D = 1, 2, 4, 8
x = torch.randn(B, H_kv, T, D)
print(f'Input  shape : {x.shape}   (2 KV heads)')

out = repeat_kv(x, num_repeats=2)
print(f'Output shape : {out.shape}  (4 KV heads after broadcast)')

# Verify data correctness — head 0 repeated into positions 0 & 1
assert torch.allclose(out[:, 0], x[:, 0]) and torch.allclose(out[:, 1], x[:, 0])
assert torch.allclose(out[:, 2], x[:, 1]) and torch.allclose(out[:, 3], x[:, 1])
print('✓ Data correctness verified — each KV head serves its assigned Q heads.')

---
## Section 3 — Rotary Positional Embedding (RoPE)

Instead of adding positional encodings to the input, RoPE *rotates* the Q and K
vectors. Relative position is then captured naturally in the dot product —
this is the approach used in Llama 2/3, GPT-NeoX, and Mistral.

$$q_m' = q_m \cdot e^{im\theta}, \quad k_n' = k_n \cdot e^{in\theta}$$
$$\Rightarrow \langle q_m', k_n' \rangle \text{ depends only on } (m - n)$$

In [ ]:
rope = RotaryEmbedding(head_dim=64, max_seq_len=128)

q = torch.randn(1, 4, 32, 64)   # (B, H, T, D_head)
k = torch.randn(1, 4, 32, 64)

q_rot, k_rot = rope(q, k, seq_len=32)

print(f'Q shape before RoPE : {q.shape}')
print(f'Q shape after  RoPE : {q_rot.shape}  (unchanged — rotation is in-place)')
print(f'Max absolute change in Q: {(q - q_rot).abs().max():.4f}')

---
## Section 4 — Full `GroupedQueryAttention` Module

The module supports three configurations controlled by `num_kv_groups`:

| `num_kv_groups` | Equivalent to |
|----------------|---------------|
| `== num_heads` | MHA (full attention) |
| `1 < x < num_heads` | GQA |
| `== 1` | MQA (extreme) |

In [ ]:
configs = [
    ('MHA  ', dict(num_heads=8, num_kv_groups=8)),
    ('GQA  ', dict(num_heads=8, num_kv_groups=2)),
    ('MQA  ', dict(num_heads=8, num_kv_groups=1)),
]

x = torch.randn(1, 20, 512)

for name, kw in configs:
    layer = GroupedQueryAttention(
        d_in=512, d_out=512, use_rope=True, causal=True, **kw
    ).eval()
    with torch.no_grad():
        out = layer(x)
    kv_params = sum(p.numel() for p in [layer.k_proj.weight, layer.v_proj.weight])
    print(f'{name}  H_KV={kw["num_kv_groups"]:>2}  '
          f'output={tuple(out.shape)}  KV params={kv_params:>8,}')

In [ ]:
# Verify causal masking: future tokens must not influence past positions
causal_layer = GroupedQueryAttention(
    d_in=64, d_out=64, num_heads=2, num_kv_groups=1,
    causal=True, max_seq_len=16
).eval()

x_full   = torch.randn(1, 8, 64)
x_masked = x_full.clone()
x_masked[:, 5:] = 0.0   # zero out positions 5-7

with torch.no_grad():
    out_full   = causal_layer(x_full)
    out_masked = causal_layer(x_masked)

identical = torch.allclose(out_full[:, :5], out_masked[:, :5], atol=1e-5)
print(f'Causal mask holds: {identical}  '
      f'(positions 0-4 are identical regardless of tokens 5-7)')

---
## Section 5 — Model Surgery: MHA → GQA via Mean-Pooling

From the GQA paper (Section 2.1):
> *"The projection matrices for key and value heads are mean pooled into single
> projection matrices, which we find works better than selecting a single key
> and value head or randomly initializing new key and value heads from scratch."*

This lets us **convert a pretrained MHA checkpoint** into GQA without training
from scratch — just a small amount of uptraining afterward.

In [ ]:
# Manual verification: group_size=2, heads=[1, 3] → average=2, heads=[5, 7] → average=6
w = torch.tensor([[1.0, 3.0, 5.0, 7.0]])
compressed = convert_mha_to_gqa(w, group_size=2)
print(f'Original   : {w.tolist()}')
print(f'Compressed : {compressed.tolist()}  ← mean of [1,3]=2 and [5,7]=6')

# Real layer example: compress 32 heads → 8 heads
layer  = nn.Linear(128, 512, bias=False)   # 32 heads × 16 head_dim
w_in   = layer.weight.t()                  # (128, 512) — (D_in, D_out)
w_gqa  = convert_mha_to_gqa(w_in, group_size=4)
print(f'\nMHA weight : {w_in.shape}  (128 in, 512 = 32 heads × 16 head_dim)')
print(f'GQA weight : {w_gqa.shape}  (128 in, 128 = 8 heads × 16 head_dim)  [4x smaller]')

---
## Section 6 — Latency Benchmark: MHA vs GQA vs MQA

In [ ]:
results = run_benchmark(
    d_model=512,
    num_heads=8,
    seq_len=256,
    batch_size=1,
    use_rope=True,
    causal=True,
    device='cpu',
    warmup=5,
    repeats=50,
)

In [ ]:
# Optional: plot results if matplotlib is available
try:
    import matplotlib.pyplot as plt
    names     = [r['name'] for r in results]
    latencies = [r['mean_ms'] for r in results]
    errors    = [r['std_ms'] for r in results]

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(names, latencies, yerr=errors, capsize=5,
                  color=['#4C72B0', '#55A868', '#C44E52'])
    ax.set_ylabel('Latency (ms)')
    ax.set_title('Forward-pass latency: MHA vs GQA vs MQA\n(d_model=512, seq_len=256, CPU)')
    for bar, lat in zip(bars, latencies):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{lat:.1f}ms', ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    plt.savefig('../assets/latency_comparison.png', dpi=150)
    plt.show()
    print('Chart saved to assets/latency_comparison.png')
except ImportError:
    print('matplotlib not installed — skipping chart.')